In [1]:
import torch

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
image_path=r"D:\Ml Dl\Project\LunaCraterNet\artifacts\data_ingestion\dataset\LU3M6TGT_yolo_format\train\images"
label_path=r"D:\Ml Dl\Project\LunaCraterNet\artifacts\data_ingestion\dataset\LU3M6TGT_yolo_format\train\labels"

In [5]:
import os
os.listdir(image_path)[:5]

['-0.28360998131250864,1.0882708585247933,-10.147301308123556,-8.775420468286237.png',
 '-0.28360998131250864,1.0882708585247933,-11.519182147960857,-10.147301308123556.png',
 '-0.28360998131250864,1.0882708585247933,-14.262943827635459,-12.89106298779817.png',
 '-0.28360998131250864,1.0882708585247933,-18.378586347147394,-17.006705507310105.png',
 '-0.28360998131250864,1.0882708585247933,-21.122348026822014,-19.75046718698471.png']

In [6]:
os.listdir(label_path)[:5]

['-0.28360998131250864,1.0882708585247933,-10.147301308123556,-8.775420468286237.txt',
 '-0.28360998131250864,1.0882708585247933,-11.519182147960857,-10.147301308123556.txt',
 '-0.28360998131250864,1.0882708585247933,-14.262943827635459,-12.89106298779817.txt',
 '-0.28360998131250864,1.0882708585247933,-18.378586347147394,-17.006705507310105.txt',
 '-0.28360998131250864,1.0882708585247933,-21.122348026822014,-19.75046718698471.txt']

In [1]:
img_name ='-0.28360998131250864,1.0882708585247933,-10.147301308123556,-8.775420468286237.png'
label_name = img_name.replace(".png", ".txt")
label_name

'-0.28360998131250864,1.0882708585247933,-10.147301308123556,-8.775420468286237.txt'

In [14]:
from PIL import Image
import numpy as np
img_name = os.listdir(image_path)[0]
path=os.path.join(img_name,image_path)
with Image.open(path) as img:
    img=np.array(img)

PermissionError: [Errno 13] Permission denied: 'D:\\Ml Dl\\Project\\LunaCraterNet\\artifacts\\data_ingestion\\dataset\\LU3M6TGT_yolo_format\\train\\images'

In [52]:
def random_rotate(image, label):
    k = random.choice([0, 1, 2, 3]) 
    
    image = np.rot90(image, k, axes=(0, 1))
    label = np.rot90(label, k, axes=(0, 1))
    
    
    return image, label

def load_yolo_label(label_path):
    boxes = []
    with open(label_path, "r") as f:
        for line in f.readlines():
            cls, x, y, w, h = map(float, line.split())
            boxes.append([cls, x, y, w, h])
    return np.array(boxes)


from torch.utils.data import Dataset,DataLoader
class ImageDataset(Dataset):
    def __init__(self,path):
        self.image_path=os.path.join(path,"images")
        self.label_path=os.path.join(path,"labels")
       
        
        self.image_files = sorted(os.listdir(self.image_path))
        self.label_files = sorted(os.listdir(self.label_path))


        # self.augmentation=augmentation

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        cur_image_path=os.path.join(self.image_path,self.image_files[idx])
        cur_label_path=os.path.join(self.label_path,self.label_files[idx])
        
        with Image.open(cur_image_path) as img:
            image=np.array(img)

        label=load_yolo_label(cur_label_path)
        label=np.array(label)
          
        


        new_image = torch.from_numpy(image).float().permute(2,0,1) / 255.0
        new_label = torch.from_numpy(label).float()
        
        return new_image, new_label

In [53]:
path=r"D:\Ml Dl\Project\LunaCraterNet\artifacts\data_ingestion\dataset\LU3M6TGT_yolo_format\train"
dataset = ImageDataset(
    path=path
)

def yolo_collate_fn(batch):
    images, labels = zip(*batch)
    images = torch.stack(images, 0)
    # Return labels as a list of tensors instead of one stacked tensor
    return images, list(labels)

# Pass this to your DataLoader
loader = DataLoader(dataset, batch_size=4, collate_fn=yolo_collate_fn)

In [58]:
ip,op= next(iter(loader))
ip.shape

torch.Size([4, 3, 416, 416])

In [61]:
len(op)

4

In [63]:
op[0].shape

torch.Size([198, 5])

In [24]:
boxes = []
label_path=r"D:\Ml Dl\Project\LunaCraterNet\artifacts\data_ingestion\dataset\LU3M6TGT_yolo_format\train\labels\-0.28360998131250864,1.0882708585247933,-3.287897108937001,-1.916016269099713.txt"
with open(label_path, "r") as f:
    for line in f.readlines():
        cls, x, y, w, h = map(float, line.split())
        boxes.append([cls, x, y, w, h])

In [32]:
boxes=np.array(boxes)
boxes.shape

(265, 5)